# Setup dan Simpan Library

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

import sys
sys.path.insert(0, os.path.abspath('..'))

INDEX_DIR = '../index'
os.makedirs(INDEX_DIR, exist_ok=True)

print("✅ Semua library berhasil diimport")

✅ Semua library berhasil diimport


# Loal Dataset

In [2]:
df = pd.read_pickle('../index/dataset_processed.pkl')
print(f"Dataset loaded: {df.shape}")
print(df[['Place_Name', 'corpus_lexical', 'corpus_semantic']].head(3))

corpus_lexical  = df['corpus_lexical'].tolist()
corpus_semantic = df['corpus_semantic'].tolist()

print(f"\nJumlah dokumen: {len(corpus_lexical)}")

Dataset loaded: (437, 16)
         Place_Name                                     corpus_lexical  \
0  Monumen Nasional  monumen nasional monumen nasional populer sing...   
1          Kota Tua  kota tua kota tua jakarta nama kota tua pusat ...   
2     Dunia Fantasi  dunia fantasi dunia fantasi dufan hibur letak ...   

                                     corpus_semantic  
0  monumen nasional monumen nasional atau yang po...  
1  kota tua kota tua di jakarta yang juga bernama...  
2  dunia fantasi dunia fantasi atau disebut juga ...  

Jumlah dokumen: 437


# Build TF-IDF Index

In [3]:
print("🔄 Membangun TF-IDF index...")

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   # unigram + bigram
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,    # log normalization
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_lexical)

print(f"✅ TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"   Vocabulary size    : {len(tfidf_vectorizer.vocabulary_)}")

# Simpan
tfidf_data = {
    'vectorizer': tfidf_vectorizer,
    'matrix': tfidf_matrix,
}
with open(f'{INDEX_DIR}/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf_data, f)
print(f"💾 TF-IDF tersimpan di {INDEX_DIR}/tfidf.pkl")

🔄 Membangun TF-IDF index...
✅ TF-IDF matrix shape: (437, 25504)
   Vocabulary size    : 25504
💾 TF-IDF tersimpan di ../index/tfidf.pkl


# Build BM25 Index

In [4]:
print("🔄 Membangun BM25 index...")

# BM25 membutuhkan tokenized corpus (list of list of words)
tokenized_corpus = [doc.split() for doc in corpus_lexical]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

print(f"✅ BM25 index selesai")
print(f"   Jumlah dokumen: {bm25.corpus_size}")

with open(f'{INDEX_DIR}/bm25.pkl', 'wb') as f:
    pickle.dump(bm25, f)
print(f"💾 BM25 tersimpan di {INDEX_DIR}/bm25.pkl")

🔄 Membangun BM25 index...
✅ BM25 index selesai
   Jumlah dokumen: 437
💾 BM25 tersimpan di ../index/bm25.pkl


# Build Sentences Transformers Embeddings

In [5]:
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
# Alternatif: 'distiluse-base-multilingual-cased-v2' (lebih akurat, lebih berat)

print(f"🔄 Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print(f"✅ Model loaded")

print(f"\n🔄 Encoding {len(corpus_semantic)} dokumen...")
print("   (Proses ini mungkin memakan waktu beberapa menit...)")

embeddings = model.encode(
    corpus_semantic,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # normalize untuk cosine similarity via inner product
)

print(f"\n✅ Embeddings shape: {embeddings.shape}")
print(f"   Dtype: {embeddings.dtype}")

np.save(f'{INDEX_DIR}/embeddings.npy', embeddings)
print(f"💾 Embeddings tersimpan di {INDEX_DIR}/embeddings.npy")

🔄 Loading model: paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

d:\University\Kuliah\6th semester\STKI\tugasSTKI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded

🔄 Encoding 437 dokumen...
   (Proses ini mungkin memakan waktu beberapa menit...)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


✅ Embeddings shape: (437, 384)
   Dtype: float32
💾 Embeddings tersimpan di ../index/embeddings.npy


# Build FAISS Index

In [6]:
print("🔄 Membangun FAISS index...")

embeddings = np.load(f'{INDEX_DIR}/embeddings.npy')

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)   # Inner Product = cosine (karena normalized)
faiss_index.add(embeddings.astype('float32'))

print(f"✅ FAISS index selesai")
print(f"   Dimensi vektor : {dimension}")
print(f"   Jumlah vektor  : {faiss_index.ntotal}")

faiss.write_index(faiss_index, f'{INDEX_DIR}/dense.faiss')
print(f"💾 FAISS tersimpan di {INDEX_DIR}/dense.faiss")

🔄 Membangun FAISS index...
✅ FAISS index selesai
   Dimensi vektor : 384
   Jumlah vektor  : 437
💾 FAISS tersimpan di ../index/dense.faiss


# Verifikasi Semua Index

In [7]:
files = ['tfidf.pkl', 'bm25.pkl', 'embeddings.npy', 'dense.faiss', 'dataset_processed.pkl']
print("=== Verifikasi Index Files ===")
for f in files:
    path = f'{INDEX_DIR}/{f}'
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"  ✅ {f:<30} ({size:.1f} MB)")

print("\n🎉 Semua index berhasil dibuat! Lanjut ke notebook 03.")

=== Verifikasi Index Files ===
  ✅ tfidf.pkl                      (1.1 MB)
  ✅ bm25.pkl                       (0.3 MB)
  ✅ embeddings.npy                 (0.6 MB)
  ✅ dense.faiss                    (0.6 MB)
  ✅ dataset_processed.pkl          (1.2 MB)

🎉 Semua index berhasil dibuat! Lanjut ke notebook 03.
